# 4.1 — Retrouver les passages et produire Q0

Section 4.1 du livret :

1. Indexer les passages (ex. BM25) et classer selon la question.
2. Sauvegarder les 5 premiers passages (rangs, scores) pour toutes les questions DEV puis TEST.
3. Produire Q0 : premier passage renvoyé tel quel, avec son identifiant, sans reformulation ni abstention (une question sans réponse produit aussi un classement).

Si le contexte dépasse la capacité du LLM, fixer une sélection reproductible sur DEV et sauvegarder les passages réellement transmis.

**Protocole.** Seul DEV est interrogé ici : TEST reste de côté jusqu'au gel du 06/11/2026 ; ses classements et
Q0 seront produits une seule fois, avec la configuration figée, dans `5_evaluation.ipynb`. BM25 est repris du TD RAG
du cours (`src/qa_recherche.py`) ; les réglages comparés sur DEV sont le texte indexé, le prétraitement des mots (mots vides, racinisation) et les paramètres k1/b.

In [1]:
import sys, json, time
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.qa_recherche import construire_bm25, scores_bm25, premiers, mesures_question, K1, B, MOTS_VIDES

DATA = RACINE / "data"
RES = RACINE / "resultats" / "qa"
K = 5

passages = [json.loads(l) for l in (DATA / "passages.jsonl").read_text(encoding="utf-8").splitlines()]
dev = [json.loads(l) for l in (DATA / "questions_dev.jsonl").read_text(encoding="utf-8").splitlines()]
ids = [p["passage_id"] for p in passages]
print(len(passages), "passages ;", len(dev), "questions DEV dont", sum(q["answerable"] for q in dev), "avec réponse")

413 passages ; 20 questions DEV dont 16 avec réponse


## Indexation BM25 et réglages sur DEV

Les questions et les passages ne sont pas modifiés : le prétraitement ne sert qu'au calcul des scores, appliqué de la
même façon aux passages (indexation) et aux questions (recherche). Q0 et les LLM reçoivent toujours le texte d'origine.

- **Texte indexé** : texte du passage seul, ou titre de section (`localisation.section`) + texte.
- **Mots vides** : liste française de NLTK, **sans les négations** (ne, pas, sans, aucun, jamais, ni, rien, non,
  plus) ; les nombres sont toujours conservés.
- **Racinisation** : Snowball français (« infections » → « infect », « douleurs » → « douleur »).

Critère de choix : Rappel@5 strict, puis couverture des éléments, puis Hit@5 ; à égalité, la configuration la plus simple.

In [2]:
TEXTES = {
    "texte": [p["texte"] for p in passages],
    "section + texte": [f'{p["localisation"]["section"]} {p["texte"]}' for p in passages],
}
PRETRAITEMENTS = {"aucun": (False, False), "mots vides": (True, False), "racines": (False, True),
                  "mots vides + racines": (True, True)}
INDEX = {(t, pt): construire_bm25(textes, *PRETRAITEMENTS[pt]) for t, textes in TEXTES.items() for pt in PRETRAITEMENTS}

def classer(question, variante, k=K, k1=K1, b=B):
    """Top-k (identifiant, rang, score) pour une question."""
    return [{"passage_id": ids[i], "rang": r, "score": round(s, 4)}
            for r, (i, s) in enumerate(premiers(scores_bm25(question, INDEX[variante], k1, b), k), start=1)]

def evaluer(variante, k1=K1, b=B):
    lignes = []
    for q in dev:
        if not q["answerable"]:
            continue
        classement = [c["passage_id"] for c in classer(q["question"], variante, len(passages), k1, b)]
        lignes.append({"question_id": q["question_id"], **mesures_question(q, classement, K),
                       "rappel@10": sum(p in classement[:10] for p in q["evidence_ids"]) / len(q["evidence_ids"])})
    return pd.DataFrame(lignes)

MESURES = [f"hit@{K}", f"rappel@{K}", f"hit@{K}_equiv", f"rappel@{K}_equiv", f"couverture_elements@{K}", "rappel@10"]
details = {v: evaluer(v) for v in INDEX}
comparaison = pd.DataFrame([{"texte indexé": t, "prétraitement": pt, **details[(t, pt)][MESURES].mean().round(3)}
                            for (t, pt) in INDEX])
comparaison.to_csv(RES / "recherche_dev_variantes.csv", index=False)
comparaison

,texte indexé,prétraitement,hit@5,rappel@5,hit@5_equiv,rappel@5_equiv,couverture_elements@5,rappel@10
0,texte,aucun,0.688,0.625,0.750,0.688,0.677,0.781
1,texte,mots vides,0.750,0.688,0.812,0.750,0.740,0.781
2,texte,racines,0.875,0.781,0.875,0.812,0.823,0.812
3,texte,mots vides + racines,0.875,0.781,0.875,0.812,0.823,0.812
4,section + texte,aucun,0.750,0.688,0.812,0.719,0.719,0.781
5,section + texte,mots vides,0.750,0.719,0.812,0.750,0.740,0.781
6,section + texte,racines,0.812,0.781,0.812,0.781,0.792,0.844
7,section + texte,mots vides + racines,0.812,0.750,0.812,0.750,0.760,0.812


In [3]:
# Rang de la première preuve, par question et par configuration (> 5 : preuve absente du top-5).
pd.DataFrame({f"{t} | {pt}": details[(t, pt)].set_index("question_id").rang_premiere_preuve for (t, pt) in INDEX}).T

question_id,Q001,Q002,Q003,Q004,Q006,Q007,Q008,Q009,Q010,Q011,Q012,Q013,Q014,Q015,Q016,Q017
texte | aucun,1,3,2,1,1,1,5,1,17,3,15,1,10,1,91,6
texte | mots vides,1,3,2,1,1,1,5,1,11,2,19,1,8,1,69,1
texte | racines,1,3,1,1,1,1,5,1,2,1,22,1,5,1,140,2
texte | mots vides + racines,1,3,1,1,1,1,3,1,1,1,25,1,5,1,112,1
section + texte | aucun,1,3,1,1,2,1,3,1,23,2,17,1,10,1,17,4
section + texte | mots vides,1,3,1,1,1,1,5,1,16,3,21,1,9,1,13,1
section + texte | racines,1,4,1,1,3,1,5,1,1,1,23,1,6,1,32,2
section + texte | mots vides + racines,1,3,1,1,1,1,4,1,1,1,27,1,8,1,22,1


In [4]:
ordre = comparaison.assign(simplicite=comparaison["prétraitement"].map({"aucun": 0, "mots vides": 1, "racines": 1, "mots vides + racines": 2})
                           + (comparaison["texte indexé"] != "texte"))
meilleure = ordre.sort_values([f"rappel@{K}", f"couverture_elements@{K}", f"hit@{K}", "simplicite"],
                              ascending=[False, False, False, True]).iloc[0]
VARIANTE = (meilleure["texte indexé"], meilleure["prétraitement"])
print("Configuration retenue :", VARIANTE)

Configuration retenue : ('texte', 'racines')


In [5]:
# Paramètres BM25 sur la configuration retenue : on ne change les valeurs du TD (1,5 ; 0,75) que pour un gain net.
grille = pd.DataFrame([{"k1": k1, "b": b, **evaluer(VARIANTE, k1, b)[[f"rappel@{K}", f"hit@{K}"]].mean().round(3)}
                       for k1 in [0.6, 0.9, 1.2, 1.5, 1.8, 2.4] for b in [0.3, 0.5, 0.75, 0.9, 1.0]])
grille.to_csv(RES / "recherche_dev_grille_k1_b.csv", index=False)
print("Rappel@5 maximal de la grille :", grille[f"rappel@{K}"].max(), "; valeurs du TD :",
      grille.query("k1 == 1.5 and b == 0.75")[f"rappel@{K}"].item())
grille.pivot(index="k1", columns="b", values=f"rappel@{K}")

Rappel@5 maximal de la grille : 0.781 ; valeurs du TD : 0.781


b,0.30,0.50,0.75,0.90,1.00
k1,,,,,
0.6,0.781,0.781,0.719,0.719,0.719
0.9,0.781,0.781,0.719,0.719,0.719
1.2,0.781,0.781,0.781,0.781,0.719
1.5,0.781,0.781,0.781,0.781,0.719
1.8,0.781,0.781,0.781,0.719,0.719
2.4,0.781,0.781,0.719,0.719,0.719


In [6]:
print(f"REC-v2 : BM25 « {VARIANTE[0]} », prétraitement « {VARIANTE[1]} », k1={K1}, b={B}, k={K}")

REC-v2 : BM25 « texte », prétraitement « racines », k1=1.5, b=0.75, k=5


## Top-5 passages par question (DEV)

In [7]:
debut = time.perf_counter()
classements = {}
for q in dev:
    t0 = time.perf_counter()
    classements[q["question_id"]] = (classer(q["question"], VARIANTE), time.perf_counter() - t0)
print(f"Recherche : {time.perf_counter() - debut:.2f} s pour {len(dev)} questions")

pd.DataFrame([{"question_id": qid, "avec_reponse": next(q["answerable"] for q in dev if q["question_id"] == qid),
               **{f"rang {c['rang']}": f"{c['passage_id']} ({c['score']:.1f})" for c in top}}
              for qid, (top, _) in classements.items()])

Recherche : 0.07 s pour 20 questions


,question_id,avec_reponse,rang 1,rang 2,rang 3,rang 4,rang 5
0,Q001,True,DOC001_P105 (24.7),DOC004_P005 (16.9),DOC004_P003 (15.5),DOC004_P002 (14.0),DOC004_P001 (13.3)
1,Q002,True,DOC007_P005 (18.1),DOC007_P004 (16.6),DOC001_P084 (14.1),DOC005_P016 (14.0),DOC001_P083 (11.2)
2,Q003,True,DOC002_P003 (27.6),DOC001_P142 (24.7),DOC003_P008 (24.7),DOC008_P002 (21.7),DOC002_P006 (17.5)
3,Q004,True,DOC002_P010 (25.4),DOC002_P009 (23.9),DOC002_P001 (14.9),DOC002_P040 (14.3),DOC002_P006 (14.2)
4,Q005,False,DOC003_P011 (15.8),DOC001_P043 (10.8),DOC005_P013 (10.3),DOC008_P009 (9.9),DOC001_P148 (9.1)
5,Q006,True,DOC009_P003 (10.3),DOC001_P073 (9.7),DOC001_P102 (8.9),DOC001_P091 (7.5),DOC005_P004 (7.4)
6,Q007,True,DOC008_P014 (22.7),DOC005_P013 (15.7),DOC001_P004 (13.5),DOC008_P002 (11.8),DOC008_P012 (11.7)
7,Q008,True,DOC005_P013 (14.0),DOC001_P031 (13.9),DOC008_P001 (13.6),DOC001_P025 (12.7),DOC008_P014 (12.2)
8,Q009,True,DOC001_P124 (12.6),DOC005_P013 (12.5),DOC001_P122 (11.6),DOC001_P100 (10.0),DOC001_P080 (9.8)
9,Q010,True,DOC001_P088 (14.8),DOC007_P009 (14.2),DOC001_P044 (14.0),DOC001_P143 (13.7),DOC003_P009 (13.7)


## Vérification : les preuves DEV sont-elles retrouvées dans le top-5 ?

Hit@5 et Rappel@5 suivent l'annexe B.3 du guide (preuves `evidence_ids`, questions avec réponse seulement).
Les colonnes `_equiv` comptent aussi les passages équivalents d'un autre document ; la couverture des éléments
indique si tous les éléments d'une réponse complexe sont disponibles dans le top-5.

In [8]:
detail = evaluer(VARIANTE).drop(columns=["rappel@10"])
detail.insert(1, "question", detail.question_id.map({q["question_id"]: q["question"] for q in dev}))
detail.insert(2, "nb_preuves", detail.question_id.map({q["question_id"]: len(q["evidence_ids"]) for q in dev}))
detail.to_csv(RES / "recherche_dev.csv", index=False)

resume = detail.drop(columns=["question_id", "question", "nb_preuves", "rang_premiere_preuve"]).mean().round(3)
print(f"DEV, {len(detail)} questions avec réponse, BM25 {VARIANTE} :")
print(resume.to_string())
detail.drop(columns=["question"])

DEV, 16 questions avec réponse, BM25 ('texte', 'racines') :
hit@5                    0.875
rappel@5                 0.781
hit@5_equiv              0.875
rappel@5_equiv           0.812
couverture_elements@5    0.823


,question_id,nb_preuves,hit@5,rappel@5,hit@5_equiv,rappel@5_equiv,couverture_elements@5,rang_premiere_preuve
0,Q001,2,1,1.0,1,1.0,1.000000,1
1,Q002,2,1,0.5,1,1.0,1.000000,3
2,Q003,2,1,1.0,1,1.0,1.000000,1
3,Q004,2,1,1.0,1,1.0,1.000000,1
4,Q006,1,1,1.0,1,1.0,1.000000,1
5,Q007,1,1,1.0,1,1.0,1.000000,1
6,Q008,1,1,1.0,1,1.0,1.000000,5
7,Q009,1,1,1.0,1,1.0,1.000000,1
8,Q010,2,1,0.5,1,0.5,0.666667,2
9,Q011,1,1,1.0,1,1.0,1.000000,1


In [9]:
# Questions dont au moins une preuve manque au top-5 : premier passage renvoyé et preuves attendues.
textes = {p["passage_id"]: p["texte"] for p in passages}
manques = detail[detail[f"rappel@{K}"] < 1]
for _, ligne in manques.iterrows():
    q = next(q for q in dev if q["question_id"] == ligne.question_id)
    top = [c["passage_id"] for c in classements[q["question_id"]][0]]
    print(f"── {q['question_id']} — {q['question']}")
    print(f"   preuves : {q['evidence_ids']} ; top-5 : {top} ; rang de la 1re preuve : {ligne.rang_premiere_preuve}")
    print(f"   rang 1 : {textes[top[0]][:160]}…\n")

── Q002 — Combien de temps durent en général les nausées et les vomissements après une séance de chimiothérapie ?
   preuves : ['DOC001_P084', 'DOC001_P085'] ; top-5 : ['DOC007_P005', 'DOC007_P004', 'DOC001_P084', 'DOC005_P016', 'DOC001_P083'] ; rang de la 1re preuve : 3
   rang 1 : La radiothérapie utilise des rayons qui, lorsqu'ils sont ciblés sur l'abdomen ou le bassin, peuvent irriter les organes du système digestif comme l'estomac, le …

── Q010 — Que manger et que boire en cas de diarrhée pendant la chimiothérapie, et quand faut-il alerter son médecin ?
   preuves : ['DOC007_P008', 'DOC007_P009'] ; top-5 : ['DOC001_P088', 'DOC007_P009', 'DOC001_P044', 'DOC001_P143', 'DOC003_P009'] ; rang de la 1re preuve : 2
   rang 1 : Quelques conseils alimentaires adaptés diminuent les nausées et les vomissements : • privilégier les aliments froids ou tièdes qui sont moins odorants que les a…

── Q012 — Mon conjoint va être nourri par sonde à la maison : comment doit-il être installé, combien 

## Production de Q0 (DEV) → `resultats/qa/predictions_q0_dev.jsonl`

Q0 renvoie le premier passage tel quel, avec son identifiant, sans reformulation ni abstention : une question sans
réponse reçoit aussi un passage. `data/predictions_qa.jsonl` est réservé aux sorties TEST produites après le gel.

In [10]:
with open(RES / "predictions_q0_dev.jsonl", "w", encoding="utf-8") as f:
    for q in dev:
        top, duree = classements[q["question_id"]]
        f.write(json.dumps({
            "configuration": "Q0", "question_id": q["question_id"], "partition": "dev",
            "recherche": f"REC-v2 : BM25 {VARIANTE[0]}, {VARIANTE[1]} (k1={K1}, b={B})",
            "passages_retrouves": top, "passages_transmis": [top[0]["passage_id"]],
            "reponse": f'{textes[top[0]["passage_id"]]} [{top[0]["passage_id"]}]',
            "duree_secondes": round(duree, 4), "echec": None,
        }, ensure_ascii=False) + "\n")
print(len(dev), "réponses Q0 écrites")
pd.read_json(RES / "predictions_q0_dev.jsonl", lines=True)[["question_id", "passages_transmis", "reponse"]].head()

20 réponses Q0 écrites


,question_id,passages_transmis,reponse
0,Q001,[DOC001_P105],Tous les médicaments de chimiothérapie n’entra...
1,Q002,[DOC007_P005],"La radiothérapie utilise des rayons qui, lorsq..."
2,Q003,[DOC002_P003],"Au total, neuf soins de support sont considéré..."
3,Q004,[DOC002_P010],Ces bilans et consultations peuvent être presc...
4,Q005,[DOC003_P011],Certains de ces effets indésirables peuvent êt...
